# Data Exploration and Preparation

## Dataset 3: COVID-FAQ / COUGH Dataset (SunLab, Ohio State University)

[Dataset3](https://github.com/sunlab-osu/covid-faq/tree/main/data)

### Data Description

#### **Dataset Origin and Context**

The COUGH (COVID-19 FAQ Retrieval) dataset is a well-established benchmark dataset designed specifically for FAQ retrieval and question answering tasks in the COVID-19 domain. It was introduced to support research in information retrieval, semantic matching, and public health question answering, particularly during the COVID-19 pandemic.
The FAQ_Bank.csv file represents the complete FAQ repository, containing 15,919 expert-curated FAQ items collected from authoritative health and government sources such as healthcare agencies, public health departments, and official medical organisations. Unlike conversational or forum-based data, this dataset focuses on non-forum, verified FAQ content, making it especially suitable for high-trust information systems.

#### **Dataset Structure**

The dataset consists of 9 columns, structured as follows:
- Column Name: Description
- index	Unique: identifier for each FAQ item
- url: Source URL of the FAQ
- question: COVID-19 related public health question
- original answer: Raw answer text (HTML-rich)
- answer wLink: Cleaned answer text with hyperlinks
- source: Publishing organisation (e.g., AHCCCS)
- language:	Language code (primarily English – en)
- type: Entry type (e.g., Question)

The dataset follows a Question–Answer (QA) structure, which aligns directly with the requirements of Retrieval-Augmented Generation (RAG) pipelines.

#### **Appropriateness for This Study**

This dataset is highly appropriate for your research for several reasons:
- Public Health Focus
  - All FAQs are framed for general users, not clinicians, aligning perfectly with your study’s emphasis on lay audiences.
- High Retrieval Complexity
  - Large FAQ bank (15k+ items)
  - Semantically overlapping questions
    → Ideal for evaluating advanced retrieval, re-ranking, and query optimisation
- Benchmark Relevance
  - COUGH is widely used in FAQ retrieval research, strengthening the academic credibility of your evaluation.
- RAG Compatibility
  - Clean QA pairs
  - Source attribution via URLs
  - Suitable for chunk-level citation grounding


### Statistical & Exploratory Analysis

In [1]:
# Import Required Libraries

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from typing import List, Dict

In [2]:
file_path3 = "/content/drive/MyDrive/MS-LJMU/Data/covid-faq-main/data/FAQ_Bank.csv"

In [3]:
df3 = pd.read_csv(file_path3)

In [4]:
df3.head()

,index,url,question,original answer,answer,answer wLink,source,language,type
0,0,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Does AHCCCS have a centralized resource for m...,"<dd><b>Answer: </b>Yes, the AHCCCS <a href=""/P...","Yes, the AHCCCS Medical Coding Resources webp...","Answer: Yes, the AHCCCS Medical Coding Resourc...",AHCCCS,en,Question
1,1,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Are there billing codes available for COVID-1...,<dd><b>Answer: </b>Yes. The Centers for Medica...,Yes. The Centers for Medicare & Medicaid Serv...,Answer: Yes. The Centers for Medicare & Medica...,AHCCCS,en,Question
2,2,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Will AHCCCS issue guidance regarding prior au...,<dd><b>Answer: </b>Prior authorization is not ...,Prior authorization is not permitted for COVI...,Answer: Prior authorization is not permitted f...,AHCCCS,en,Question
3,3,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Is there a claims modifier for services relat...,"<dd><b>Answer: </b>Yes, AHCCCS has designated ...","Yes, AHCCCS has designated the CR modifier to...","Answer: Yes, AHCCCS has designated the CR modi...",AHCCCS,en,Question
4,4,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Does AHCCCS cover testing for COVID-19?,"<dd><b>Answer: </b>Yes, AHCCCS covers COVID-19...","Yes, AHCCCS covers COVID-19 testing. HCPCS U0...","Answer: Yes, AHCCCS covers COVID-19 testing. H...",AHCCCS,en,Question


In [5]:
df3["language"].unique()

array(['en', 'es', 'ko', 'vi', 'zh', 'ar', 'fa', 'hy', 'kr', 'ja', 'fr',
       'ru', 'cs', 'de', 'hi', 'id', 'it', 'nl', 'pt', 'th'], dtype=object)

In [7]:
df3 = df3[df3["language"]=="en"] # Taking only english Q&A

In [9]:
df3 = df3.dropna() # Drop null values

In [10]:
# Basic Dataset Inspection


print("Dataset Shape:", df3.shape)

print("Columns:", df3.columns.tolist())

df3.isnull().sum()

Dataset Shape: (9148, 9)
Columns: ['index', 'url', 'question', 'original answer', 'answer', 'answer wLink', 'source', 'language', 'type']


,0
index,0
url,0
question,0
original answer,0
answer,0
answer wLink,0
source,0
language,0
type,0


In [11]:
df3["question"]

,question
0,Does AHCCCS have a centralized resource for m...
1,Are there billing codes available for COVID-1...
2,Will AHCCCS issue guidance regarding prior au...
3,Is there a claims modifier for services relat...
4,Does AHCCCS cover testing for COVID-19?
...,...
9146,Am I High Risk for Covid19? I am a diabetic wh...
9147,Am I High Risk for Covid19? I am a diabetic wh...
9148,Am I High Risk for Covid19? I am a diabetic wh...
9149,Children and grandparents. \n \n\n ...


In [12]:
df3["answer wLink"]

,answer wLink
0,"Answer: Yes, the AHCCCS Medical Coding Resourc..."
1,Answer: Yes. The Centers for Medicare & Medica...
2,Answer: Prior authorization is not permitted f...
3,"Answer: Yes, AHCCCS has designated the CR modi..."
4,"Answer: Yes, AHCCCS covers COVID-19 testing. H..."
...,...
9146,Those who are higher risk of novel coronavirus...
9147,"Gosh, I'm sorry you went through that. I'm ve..."
9148,Do you have any remaining issues from the seps...
9149,If you have any children or grandparents artic...


### Visualisations and Insights

# Embedding Preparation

In [21]:
EMBEDDING_MODEL = "text-embedding-3-large"
EMBEDDING_DIM = 3072

In [13]:
def build_embedding_text(row):
    return f"""Question: {row['question']}

Answer: {row['answer']}"""

In [14]:
!pip install langchain
!pip install openai
!pip install tiktoken

In [15]:
!pip install -U langchain-text-splitters

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

In [18]:
def chunk_faq(row):
    chunks = splitter.split_text(row["answer"])
    documents = []

    for i, chunk in enumerate(chunks):
        text = f"""Question: {row['question']}

Answer (Part {i+1}): {chunk}"""
        documents.append((text, i))

    return documents

In [19]:
from langchain_core.documents import Document

In [20]:
documents = []

for _, row in df3.iterrows():
    chunks = chunk_faq(row)
    for text, chunk_id in chunks:
        metadata = {
            "faq_id": row["index"],
            "source": row["source"],
            "url": row["url"],
            "language": row["language"],
            "type": row["type"],
            "chunk_id": chunk_id,
            "dataset": "COUGH_FAQ"
        }

        documents.append(
            Document(
                page_content=text,
                metadata=metadata
            )
        )

In [22]:
import tiktoken

encoding = tiktoken.encoding_for_model(EMBEDDING_MODEL)

def count_tokens(text: str) -> int:
    """
    Count tokens for a given text using tiktoken.
    """
    return len(encoding.encode(text))


def add_token_metadata(documents):
    """
    Adds token count to each Document's metadata.
    """
    for doc in documents:
        doc.metadata["embedding_tokens"] = count_tokens(doc.page_content)
    return documents

In [23]:
documents = add_token_metadata(documents)

In [26]:
documents[5]

Document(metadata={'faq_id': 1, 'source': 'AHCCCS', 'url': 'https://www.azahcccs.gov/AHCCCS/AboutUs/covid19FAQ.html', 'language': 'en', 'type': 'Question', 'chunk_id': 4, 'dataset': 'COUGH_FAQ', 'embedding_tokens': 63}, page_content='Question:  Are there billing codes available for COVID-19 testing outside of Centers for Disease Control and Prevention (CDC) testing laboratories?\n\nAnswer (Part 5): that would otherwise be identified by U0002. Neither U0003 nor U0004 should be used for tests that detect COVID-19 antibodies.')

# Data Indexing (Embeddings)

### FAISS: Facebook AI Similarity Search

[Link](https://docs.langchain.com/oss/python/integrations/vectorstores/faiss)

In [28]:
from openai import OpenAI
from google.colab import userdata

In [29]:
api_key = userdata.get('OPENAI_API_KEY')

In [30]:
os.environ["OPENAI_API_KEY"] = api_key

In [31]:
client = OpenAI()

In [32]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 2.9 MB/s eta 0:00:00


In [33]:
!pip install -qU langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [34]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [35]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

In [36]:
index = faiss.IndexFlatL2(EMBEDDING_DIM)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

#### Add items to vector store

In [37]:
from uuid import uuid4

In [38]:
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['69132eb2-66ad-41c3-88f7-782aa3727019',
 '1950903e-9259-459b-a627-3344321d00a4',
 'fdd5af07-1a28-4992-9600-e72467075c8d',
 '28787933-1ab9-4c63-adce-c7f0a21ba7fc',
 '4dfad708-2fa6-443b-adc5-8b96546f290a',
 '8db11a14-bcce-49df-b5b2-1d9b9d8c484d',
 '6743db54-5dff-44b7-91aa-525247d9c92e',
 'a59f2cc1-0121-4ae8-a33a-625ba24b4734',
 'dbfd662e-fbbb-417d-a94d-58c6ef3d1e9c',
 'ea5bab19-bb41-4962-bc3d-556b77a632e9',
 'e628f1e0-5e72-4613-9f4c-0afe2cfb01e4',
 '38d0289e-ef83-4318-85bd-c689f0a687cc',
 '47164fb7-db00-4a0e-86e4-674fd70d552f',
 'db63b15c-603f-4ecb-a47c-50a109377662',
 '3c715797-7a39-4769-806a-4ffe6cc1ea92',
 '8a67a447-23b7-4834-8e4a-0e5ea54903e9',
 '149fc1b2-2616-4375-8e3a-b402d2bb90e3',
 '96538bf7-d926-4b86-a2ec-1712bad2c004',
 'bdf07712-f769-481a-84e0-6efb1eca22cd',
 '3266d946-6ce9-4a43-8956-ab3cf8468fd1',
 '7bf0ba4b-5ba6-4fbd-96f0-21b77a71ebba',
 'c94c7708-37f3-4cc3-a0c8-a0b7b2ec9086',
 '4ef4526c-188d-49fb-a3ec-7291692958aa',
 'bfb75bb3-3bf6-4be4-b458-c56588d3ee90',
 '2b8a2f72-8726-

#### Saving Vector Store

In [39]:
documents[0]

Document(metadata={'faq_id': 0, 'source': 'AHCCCS', 'url': 'https://www.azahcccs.gov/AHCCCS/AboutUs/covid19FAQ.html', 'language': 'en', 'type': 'Question', 'chunk_id': 0, 'dataset': 'COUGH_FAQ', 'embedding_tokens': 55}, page_content='Question:  Does AHCCCS have a centralized resource for medical coding resources related to COVID-19?\n\nAnswer (Part 1): Yes, the AHCCCS Medical Coding Resources webpage includes a COVID-19 Medical Coding Information section and COVID-19 Emergency Medical Coding guidance.')

In [40]:
vector_store_path = os.path.join("/content/drive/MyDrive/MS-LJMU/Vector-Store", "Store-3-3072-COUGH-FAQ-ENG")

In [41]:
vector_store.save_local(vector_store_path)